<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Visitas_parques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Cargar archivo
df_parques = pd.read_csv("https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_visitas_parques.csv")

print("Registros originales:", len(df_parques))

df_parques.head()

Registros originales: 15000


,id_visita_parque,fecha,parque,pais_origen,gasto_estimado
0,1,2022-11-07,Puerta del Diablo,Canada,59
1,2,2024-02-23,Amulapa,Costa Rica,59
2,3,2021-07-09,Sunset Park,Mexico,74
3,4,2024-03-21,Puerta del Diablo,España,64
4,5,2024-03-21,Sunset Park,Costa Rica,29


**Inspección inicial**

In [2]:
print(df_parques.columns.tolist())

df_parques.info()

['id_visita_parque', 'fecha', 'parque', 'pais_origen', 'gasto_estimado']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_visita_parque  15000 non-null  int64 
 1   fecha             15000 non-null  object
 2   parque            15000 non-null  object
 3   pais_origen       15000 non-null  object
 4   gasto_estimado    15000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 586.1+ KB


**Transformar**

Conversión de fecha

In [3]:
df_parques['fecha'] = pd.to_datetime(df_parques['fecha'])

Crear atributos de tiempo

In [4]:
df_parques['anio'] = df_parques['fecha'].dt.year
df_parques['mes'] = df_parques['fecha'].dt.month
df_parques['dia'] = df_parques['fecha'].dt.day
df_parques['trimestre'] = df_parques['fecha'].dt.quarter

Limpiar textos

In [5]:
df_parques['parque'] = (
    df_parques['parque']
    .astype(str)
    .str.strip()
    .str.title()
)

In [6]:
df_parques['pais_origen'] = (
    df_parques['pais_origen']
    .astype(str)
    .str.strip()
    .str.title()
)

Eliminar duplicados

In [7]:
print("Duplicados:", df_parques.duplicated().sum())

df_parques.drop_duplicates(inplace=True)

Duplicados: 0


Validar gasto estimado

In [8]:
df_parques = df_parques[
    df_parques['gasto_estimado'] >= 0
]

Clasificar gasto turístico

In [9]:
df_parques['categoria_gasto'] = np.where(
    df_parques['gasto_estimado'] < 30,
    'Bajo',
    np.where(
        df_parques['gasto_estimado'] < 60,
        'Medio',
        'Alto'
    )
)

Verificación

In [10]:
print(df_parques.info())

print(df_parques.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_visita_parque  15000 non-null  int64         
 1   fecha             15000 non-null  datetime64[ns]
 2   parque            15000 non-null  object        
 3   pais_origen       15000 non-null  object        
 4   gasto_estimado    15000 non-null  int64         
 5   anio              15000 non-null  int32         
 6   mes               15000 non-null  int32         
 7   dia               15000 non-null  int32         
 8   trimestre         15000 non-null  int32         
 9   categoria_gasto   15000 non-null  object        
dtypes: datetime64[ns](1), int32(4), int64(2), object(3)
memory usage: 937.6+ KB
None
       id_visita_parque                       fecha  gasto_estimado  \
count      15000.000000                       15000    15000.000000   
mean        750

**Carga**

In [11]:
df_parques.to_csv(
    'dw_fact_visitas_parques.csv',
    index=False
)

print("ETL completado correctamente")

ETL completado correctamente
